In [ ]:
# =====================================================
# 0. 라이브러리 세팅
# =====================================================
def load_libraries():
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    import math
    import platform
    import ast

    pd.set_option('display.float_format', "{:.2f}".format)

    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    elif platform.system() == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    else:
        plt.rcParams['font.family'] = 'NanumGothic'

    return pd, np, plt, ast


# =====================================================
# 1. 데이터 로드
# =====================================================
def load_data(pd):
    df_port = pd.read_csv("../dataset/portfolio.csv", index_col=0)
    df_prof = pd.read_csv("../dataset/profile.csv", index_col=0)
    df_tran = pd.read_csv("../dataset/transcript.csv", index_col=0)
    df_menu = pd.read_csv("../dataset/starbucks_menu_260112.csv", index_col=0)

    print("프로모션 제공 데이터 크기:", df_port.shape)
    print("고객정보 데이터 크기:", df_prof.shape)
    print("제공 프로모션 데이터 크기:", df_tran.shape)
    print("메뉴 정보 데이터 크기:", df_menu.shape)

    return df_port, df_prof, df_tran, df_menu


# =====================================================
# 2. Portfolio 전처리
# =====================================================
def preprocess_portfolio(df_port, np):
    df_port = df_port.copy()

    # One-hot encoding
    df_port['ch_web'] = df_port['channels'].apply(lambda x: 1 if 'web' in x else 0)
    df_port['ch_email'] = df_port['channels'].apply(lambda x: 1 if 'email' in x else 0)
    df_port['ch_mobile'] = df_port['channels'].apply(lambda x: 1 if 'mobile' in x else 0)
    df_port['ch_social'] = df_port['channels'].apply(lambda x: 1 if 'social' in x else 0)
    df_port['channel_count'] = df_port[['ch_web','ch_email','ch_mobile','ch_social']].sum(axis=1)

    # 필요없는 컬럼 제거
    drop_cols = ['channels', 'Unnamed: 0']
    df_port = df_port.drop(columns=[c for c in drop_cols if c in df_port.columns])

    # 파생 변수
    # reward_ratio : 1원을 쓸 때 돌아오는 보상이 얼마인가? (가성비)
    # offer_strength : 순 혜택의 크기
    df_port['reward_ratio'] = np.where(df_port['difficulty']==0, df_port['reward'], df_port['reward']/df_port['difficulty'])
    df_port['offer_strength'] = df_port['reward'] - df_port['difficulty']

    # offer_id로 컬럼명 변경
    df_port = df_port.rename(columns={'id':'offer_id'})
    df_port['offer_id'] = df_port['offer_id'].astype(str)

    return df_port


# =====================================================
# 3. Transcript 전처리 + viewed_before_complete 플래그
# =====================================================
def preprocess_transcript(df_tran, ast):
    df_tran = df_tran.copy()

    # value 컬럼 파싱
    df_tran['value'] = df_tran['value'].apply(ast.literal_eval)
    df_tran['amount'] = df_tran['value'].str.get('amount')
    df_tran['actual_reward'] = df_tran['value'].str.get('reward')

    temp_id1 = df_tran['value'].str.get('offer id')
    temp_id2 = df_tran['value'].str.get('offer_id')
    df_tran['offer_id'] = temp_id1.fillna(temp_id2)

    # 컬럼 제거, day 파생
    df_tran.drop(columns='value', inplace=True)
    df_tran['day'] = df_tran['time'] // 24

    # ID 타입 문자열로
    df_tran['offer_id'] = df_tran['offer_id'].astype(str)
    df_tran = df_tran.rename(columns={'person':'customer_id'})
    df_tran['customer_id'] = df_tran['customer_id'].astype(str) 

    return df_tran


# =====================================================
# 4. Profile 전처리
# =====================================================
def preprocess_profile(df_prof, pd, np):
    df_prof = df_prof.copy()

    # 이상치 처리
    df_prof['age'] = df_prof['age'].replace(118, np.nan)
    df_prof['gender'] = df_prof['gender'].fillna('Unknown')
    df_prof['income'] = df_prof['income'].fillna(0)

    # 결측 그룹 flag
    df_prof['is_profile_missing'] = np.where(
        (df_prof['gender']=='Unknown') & (df_prof['income']==0) & (df_prof['age'].isna()), 1, 0
    )

    # 연령대 파생변수
    def age_group(df):
        if pd.isna(df['age']):
            return '누락'
        elif df['age'] < 20:
            return '20대 미만'
        elif df['age'] < 30:
            return '20대'
        elif df['age'] < 40:
            return '30대'
        elif df['age'] < 50:
            return '40대'
        elif df['age'] < 60:
            return '50대'
        else:
            return '60대 이상'

    df_prof['age_group'] = df_prof.apply(age_group, axis=1)

    # 연령대 × 성별 파생변수
    def group_age_gender(df):
        if df['gender'] == 'Unknown' or pd.isna(df['age']):
            return '미기입'
        elif df['gender'] == 'O':
            return 'Others'
        elif df['gender'] == 'M':
            if df['age'] < 20:   return '20세 미만 남성'
            elif df['age'] < 30: return '20대 남성'
            elif df['age'] < 40: return '30대 남성'
            elif df['age'] < 50: return '40대 남성'
            elif df['age'] < 60: return '50대 남성'
            elif df['age'] < 70: return '60대 남성'
            else:                return '60+ 남성'
        else:
            if df['age'] < 20:   return '20세 미만 여성'
            elif df['age'] < 30: return '20대 여성'
            elif df['age'] < 40: return '30대 여성'
            elif df['age'] < 50: return '40대 여성'
            elif df['age'] < 60: return '50대 여성'
            elif df['age'] < 70: return '60대 여성'
            else:                return '60+ 여성'

    df_prof['age_gender'] = df_prof.apply(group_age_gender, axis=1)

    # 수익 범주화 파생변수
    def income_group(income):
        if pd.isna(income) or income == 0:
            return '누락'
        elif income < 50000:
            return '5만 미만'
        elif income < 75000:
            return '5-7.5만'
        elif income < 100000:
            return '7.5-10만'
        else:
            return '10만 이상'

    df_prof['income_group'] = df_prof['income'].apply(income_group)
        
    # 날짜 변환
    df_prof['became_member_on'] = pd.to_datetime(df_prof['became_member_on'], format='%Y%m%d')

    # id → customer_id
    df_prof = df_prof.rename(columns={'id':'customer_id'})
    df_prof['customer_id'] = df_prof['customer_id'].astype(str)

    return df_prof


# =====================================================
# 5. 병합 (tran + port + prof)
# =====================================================
def merge_data(df_tran, df_port, df_prof):
    df_tran = df_tran.copy()
    df_port = df_port.copy()
    df_prof = df_prof.copy()

    df = df_tran.merge(df_port, on='offer_id', how='left')
    df = df.merge(df_prof, on='customer_id', how='left')

    print("Before merge:", df_tran.shape)
    print("After merge:", df.shape)
    return df


# =====================================================
# 6. Save Data
# =====================================================
def save_data(df, filepath="../dataset/preprocessed_final.csv"):
    df.to_csv(filepath, index=False)
    print(f"✅ {filepath} 저장 완료")


# =====================================================
# 7. Run Pipeline
# =====================================================
def run_pipeline():
    pd, np, plt, ast = load_libraries()
    df_port, df_prof, df_tran, df_menu = load_data(pd)

    df_port = preprocess_portfolio(df_port, np)
    df_tran = preprocess_transcript(df_tran, ast)
    df_prof = preprocess_profile(df_prof, pd, np)

    df = merge_data(df_tran, df_port, df_prof)
    save_data(df)

    return df


# =====================================================
# 실행
# =====================================================
df = run_pipeline()
df.head()

In [ ]:
df.columns